# The Gaussian Integral (Normalization Constant)

Companion notebook for: [The Gaussian Integral](https://ml-viz.vercel.app/wiki/gaussian-integral)

We numerically verify:
1. The Gaussian integral $I = \int_{-\infty}^\infty e^{-x^2/2}\,dx = \sqrt{2\pi}$
2. $\mathbb{E}[X] = \mu$ and $\text{Var}(X) = \sigma^2$ via Monte Carlo
3. How Monte Carlo approximation error decays with sample count

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'font.size': 11,
})

## 1 · Exact numerical integration

scipy.integrate.quad uses adaptive Gaussian quadrature — highly accurate for smooth functions.

In [ ]:
# The Gaussian integral I = ∫_{-∞}^{∞} e^{-x²/2} dx
I_exact, error = integrate.quad(lambda x: np.exp(-x**2 / 2), -np.inf, np.inf)
I_theory = np.sqrt(2 * np.pi)

print(f"scipy.integrate.quad:  I = {I_exact:.15f}")
print(f"sqrt(2π):                  {I_theory:.15f}")
print(f"Absolute difference:       {abs(I_exact - I_theory):.2e}")
print(f"Estimated integration err: {error:.2e}")

## 2 · Visualise the polar trick

The 2-D integrand $e^{-(x^2+y^2)/2}$ that we convert to polar coordinates.

In [ ]:
x_1d = np.linspace(-4, 4, 300)
x2d, y2d = np.meshgrid(x_1d, x_1d)
z = np.exp(-(x2d**2 + y2d**2) / 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1-D Gaussian
axes[0].fill_between(x_1d, np.exp(-x_1d**2 / 2), alpha=0.4, color='#6366f1')
axes[0].plot(x_1d, np.exp(-x_1d**2 / 2), color='#6366f1', lw=2)
axes[0].set(title=f'1-D: ∫e^{{-x²/2}} dx = {I_exact:.4f} = √(2π)',
            xlabel='x', ylabel='e^{-x²/2}')
axes[0].text(0, 0.3, f'Area = √(2π)\n≈ {I_theory:.4f}',
             ha='center', color='#f59e0b', fontsize=12)

# 2-D version (polar trick)
im = axes[1].contourf(x2d, y2d, z, levels=20, cmap='plasma')
fig.colorbar(im, ax=axes[1])
# Draw a few radii
for theta in np.linspace(0, 2*np.pi, 9)[:-1]:
    axes[1].plot([0, 3*np.cos(theta)], [0, 3*np.sin(theta)],
                 color='white', alpha=0.3, lw=1)
axes[1].set(title='2-D: e^{-(x²+y²)/2} → polar → I² = 2π', xlabel='x', ylabel='y')

plt.tight_layout()
plt.show()

## 3 · Monte Carlo estimation and error decay

We estimate $\mathbb{E}[X]$ and $\text{Var}(X)$ for $\mathcal{N}(\mu, \sigma^2)$ by sampling, and show that error decays as $1/\sqrt{N}$.

In [ ]:
rng = np.random.default_rng(0)
mu_true, sigma_true = 3.0, 2.0

Ns = np.logspace(2, 6, 30, dtype=int)
mean_errors = []
var_errors  = []

for N in Ns:
    samples = rng.normal(mu_true, sigma_true, N)
    mean_errors.append(abs(samples.mean() - mu_true))
    var_errors.append(abs(samples.var() - sigma_true**2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, errors, label, color in [
    (axes[0], mean_errors, '|E[X] - μ|', '#6366f1'),
    (axes[1], var_errors,  '|Var[X] - σ²|', '#f59e0b'),
]:
    ax.loglog(Ns, errors, 'o-', color=color, ms=4, lw=1.5, label=label)
    # 1/sqrt(N) reference
    ref = errors[0] * np.sqrt(Ns[0]) / np.sqrt(Ns)
    ax.loglog(Ns, ref, '--', color='#94a3b8', alpha=0.7, label='1/√N reference')
    ax.set(xlabel='N (samples)', ylabel='Absolute error', title=label)
    ax.legend()
    ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Monte Carlo estimation error for N(μ=3, σ²=4)', y=1.02)
plt.tight_layout()
plt.show()

## 4 · Verify the normalization constant for different σ

In [ ]:
print(f"{'σ':>5}  {'∫p(x)dx (quad)':>18}  {'error':>10}")
print("-" * 40)
for sigma in [0.1, 0.5, 1.0, 2.0, 5.0]:
    area, err = integrate.quad(
        lambda x: (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-x**2 / (2 * sigma**2)),
        -50*sigma, 50*sigma
    )
    print(f"{sigma:>5.1f}  {area:>18.15f}  {abs(area - 1):.2e}")

---

## ✏️ Your turn

### Exercise 1 — integration by substitution

Verify numerically that $\int_{-\infty}^\infty z\, \phi(z)\, dz = 0$ (the odd-function argument for $\mathbb{E}[X]=\mu$).

In [ ]:
# TODO(you): integrate z * phi(z) numerically; confirm result is ~0
phi = lambda z: (1 / np.sqrt(2*np.pi)) * np.exp(-z**2 / 2)
integrand = lambda z: z * phi(z)
# result, _ = integrate.quad(integrand, -np.inf, np.inf)
# assert abs(result) < 1e-10, f"Expected 0, got {result}"
print("Fill in the code above and uncomment the assert.")

### Exercise 2 — variance integral

Verify numerically that $\int_{-\infty}^\infty z^2\, \phi(z)\, dz = 1$.

In [ ]:
# TODO(you): integrate z^2 * phi(z); confirm result is ~1
# integrand2 = lambda z: z**2 * phi(z)
# result2, _ = integrate.quad(integrand2, -np.inf, np.inf)
# assert abs(result2 - 1.0) < 1e-10, f"Expected 1, got {result2}"
print("Fill in and uncomment the assert.")

<details>
<summary>Solutions</summary>

```python
# Exercise 1
result, _ = integrate.quad(lambda z: z * phi(z), -np.inf, np.inf)
assert abs(result) < 1e-10, f"Expected 0, got {result}"
print(f"∫z·φ(z)dz = {result:.2e}  ✓")

# Exercise 2
result2, _ = integrate.quad(lambda z: z**2 * phi(z), -np.inf, np.inf)
assert abs(result2 - 1.0) < 1e-10
print(f"∫z²·φ(z)dz = {result2:.10f}  ✓")
```
</details>